# 03 - Categorical Standardization




EDA found `Order Country` mixes English and Spanish naming (Hungría, Irlanda, Malasia, Costa de Marfil). This notebook checks for and fixes any country accidentally split across two spellings, plus resolves the `Category Id`/`Category Name` count mismatch (51 vs. 50) flagged in EDA.

## Setup

In [1]:
import pandas as pd
from pathlib import Path

INPUT_PATH = Path('../../../data/processed/orders_clean.csv')
OUTPUT_PATH = Path('../../../data/processed/orders_standardized.csv')

orders_df = pd.read_csv(INPUT_PATH)
print(f"Shape: {orders_df.shape[0]:,} rows x {orders_df.shape[1]} columns")


Shape: 65,752 rows x 19 columns


## 1. Inspect Order Country values for language inconsistency

List every unique value - with 164 countries this is short enough to eyeball directly for obvious duplicates (e.g. a country appearing once in English and once in Spanish).


In [2]:
all_countries = sorted(orders_df['Order Country'].unique())
print(f"Total unique Order Country values: {len(all_countries)}")
for c in all_countries:
    print(c)


Total unique Order Country values: 164
Afganistán
Albania
Alemania
Angola
Arabia Saudí
Argelia
Argentina
Armenia
Australia
Austria
Azerbaiyán
Bangladés
Barbados
Baréin
Belice
Benín
Bielorrusia
Bolivia
Bosnia y Herzegovina
Botsuana
Brasil
Bulgaria
Burkina Faso
Burundi
Bután
Bélgica
Camboya
Camerún
Canada
Chad
Chile
China
Chipre
Colombia
Corea del Sur
Costa Rica
Costa de Marfil
Croacia
Cuba
Dinamarca
Ecuador
Egipto
El Salvador
Emiratos Árabes Unidos
Eritrea
Eslovaquia
Eslovenia
España
Estados Unidos
Estonia
Etiopía
Filipinas
Finlandia
Francia
Gabón
Georgia
Ghana
Grecia
Guadalupe
Guatemala
Guayana Francesa
Guinea
Guinea Ecuatorial
Guinea-Bissau
Guyana
Haití
Honduras
Hong Kong
Hungría
India
Indonesia
Irak
Irlanda
Irán
Israel
Italia
Jamaica
Japón
Jordania
Kazajistán
Kenia
Kirguistán
Kuwait
Laos
Lesoto
Liberia
Libia
Lituania
Luxemburgo
Líbano
Macedonia
Madagascar
Malasia
Mali
Marruecos
Martinica
Mauritania
Moldavia
Mongolia
Montenegro
Mozambique
Myanmar (Birmania)
México
Namibia
Nepal
Nicara

**What we found - this corrects an earlier EDA assumption, not just confirms it:**

**Looking at the full list of 164 country values, `Order Country` is consistently in Spanish throughout - it is not actually a mix of English and Spanish.** The earlier EDA finding (Notebook 05) flagged names like "Hungría," "Irlanda," and "Costa de Marfil" as evidence of inconsistent localization, but seeing the complete list makes clear those are just the Spanish names - and entries that looked "English" in the EDA sample (Ghana, Canada, Portugal, Argentina, Australia, Austria, Cuba, Chile, China, etc.) are simply countries whose Spanish and English spellings happen to be identical or near identical by coincidence. Every other country (México, Alemania, Francia, Reino Unido, Estados Unidos, Países Bajos, etc.) is unambiguously Spanish.

**Practical implication: there is no language mixing problem to fix, and the `0 corrections applied` result above is the correct outcome, not a skipped step.** Since the entire column is uniformly Spanish, no country name has an English "twin" spelling sitting elsewhere in the data - so nothing is being silently split across two categories.

**One minor, unrelated formatting quirk worth noting (not a duplication risk):** `SudAfrica` (South Africa) is written as one word with no space, inconsistent with other multi word country names like `Costa de Marfil` or `Estados Unidos` which do have spaces. This is cosmetic and doesn't create a duplicate category, so it's optional to clean up - flagging it here for completeness.

**Correction to log:** the EDA finding about English/Spanish mixing in `Order Country` was based on an incomplete sample and turned out to be inaccurate - the column is consistently Spanish. 


In [5]:
country_name_corrections = {
    # 'Spanish name': 'English standard name',
    # e.g. 'Hungría': 'Hungary',
    # e.g. 'Irlanda': 'Ireland',
}

orders_df['Order Country'] = orders_df['Order Country'].replace(country_name_corrections)
print(f"Applied {len(country_name_corrections)} corrections.")
print(f"Unique countries after standardization: {orders_df['Order Country'].nunique()}")


Applied 0 corrections.
Unique countries after standardization: 164


## 2. Resolve the Category Id / Category Name mismatch

EDA found 51 unique `Category Id` values but only 50 unique `Category Name` values - check whether two IDs share one name, or one ID has an inconsistent name.


In [6]:
category_mapping_check = orders_df.groupby('Category Id')['Category Name'].nunique()
inconsistent_ids = category_mapping_check[category_mapping_check > 1]
print(f"Category Ids with more than one associated name: {len(inconsistent_ids)}")
if len(inconsistent_ids) > 0:
    display(orders_df[orders_df['Category Id'].isin(inconsistent_ids.index)][['Category Id', 'Category Name']].drop_duplicates())

name_to_id_check = orders_df.groupby('Category Name')['Category Id'].nunique()
shared_names = name_to_id_check[name_to_id_check > 1]
print(f"\nCategory Names shared by more than one Id: {len(shared_names)}")
if len(shared_names) > 0:
    display(orders_df[orders_df['Category Name'].isin(shared_names.index)][['Category Id', 'Category Name']].drop_duplicates())


Category Ids with more than one associated name: 28


,Category Id,Category Name
1,18,Men's Footwear
2,17,Accessories
4,41,Camping & Hiking
5,17,Cleats
6,48,Water Sports
...,...,...
54675,2,Men's Golf Clubs
54912,10,Shop By Sport
55756,3,As Seen on TV!
55848,30,Kids' Golf Clubs



Category Names shared by more than one Id: 24


,Category Id,Category Name
0,43,Camping & Hiking
1,18,Men's Footwear
2,17,Accessories
4,41,Camping & Hiking
5,17,Cleats
...,...,...
54675,2,Men's Golf Clubs
54912,10,Shop By Sport
55756,3,As Seen on TV!
55848,30,Kids' Golf Clubs


**What we found - this is a bigger issue than the EDA's "51 vs. 50" count suggested:**

**This is not a single one off mismatch - it's a widespread many to many inconsistency.** 28 `Category Id` values map to more than one `Category Name`, and 24 `Category Name` values are shared across more than one `Category Id`. For example, `Category Id` 17 appears mapped to both "Accessories" and "Cleats" - these are not remotely similar categories, so this isn't a spelling variation, it's a genuine identifier collision.

**Most likely explanation, checked directly in the cell below:** category IDs may be scoped *within* a department rather than globally unique - "Category 17" could mean "Accessories" in one department and something else in another. If pairing with `Department Id` resolves the ambiguity, that confirms department scoping; if inconsistencies remain even then, it's a genuine data quality issue in the source file.


### 2a. Quick check - is Category Id scoped within Department Id?

`Department Id` wasn't carried into the order level aggregation (Notebook 01 only kept the columns needed for modelling), so we reload the raw line item file just for this one check.


In [7]:
raw_check_path = Path('../../../data/raw data/DataCoSupplyChainDataset.csv')  

try:
    raw_df = pd.read_csv(raw_check_path, encoding='utf-8')
except UnicodeDecodeError:
    raw_df = pd.read_csv(raw_check_path, encoding='ISO-8859-1')

# Does pairing Category Id with Department Id give a clean 1:1 mapping to Category Name?
paired_check = raw_df.groupby(['Department Id', 'Category Id'])['Category Name'].nunique()
still_inconsistent = paired_check[paired_check > 1]

print(f"(Department Id, Category Id) pairs that still map to more than one Category Name: {len(still_inconsistent)}")
if len(still_inconsistent) > 0:
    display(still_inconsistent.head(10))
else:
    print("Every (Department Id, Category Id) pair maps to exactly one Category Name — department-scoping confirmed.")


(Department Id, Category Id) pairs that still map to more than one Category Name: 0
Every (Department Id, Category Id) pair maps to exactly one Category Name — department-scoping confirmed.


**What we found - the department  scoping theory is confirmed, cleanly:**

**Zero `(Department Id, Category Id)` pairs map to more than one `Category Name`.** Every single mismatch found earlier disappears once `Category Id` is paired with `Department Id` - this isn't a data quality problem at all, it's a normal ID scoping convention: `Category Id` values are only unique *within* a department, not globally. "Category 17" genuinely means "Accessories" in one department and "Cleats" in another, and both are valid, intentional values - not an error.

**This changes the working decision from Notebook 03's earlier finding:**
- `Category Id` **alone is not reliable and should not be used as a standalone feature** - confirmed, not just suspected.
- `Category Name` **alone is actually fine to use as is** - since it's derived directly from the correct department context in the raw data (each line item's `Category Name` is already the correct, department aware value; the confusion only appeared when looking at `Category Id` across departments). So the interim decision from earlier in this notebook (use `Category Name` as the canonical feature) turns out to be the right call, now for a confirmed reason rather than a cautious default.
- **If department level granularity is ever wanted as its own feature**, the reliable combination is `Department Id` + `Category Id` together, not `Category Id` on its own.

**Practical follow up needed:** since this check required reloading the raw line item file (`Department Id` isn't in the current aggregated dataset), no change is actually needed to Notebook 01's aggregation - we're keeping `Category Name` as the feature either way, and `Department Id` doesn't need to be added unless the group later decides department level features are worth adding for their own sake (not required by this fix).

**`DECISION_LOG.md`:** Category Id/Name mismatch fully explained-— confirmed as department scoped IDs, not a data error. `Category Name` confirmed as the reliable, canonical category feature; `Category Id` excluded as a standalone feature unless paired with `Department Id`.


## 3. Save the standardized dataset

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
orders_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH.resolve()}")


Saved to: C:\Users\Ewis\Documents\Machine_learning_project\Heads-Up_IT3091\data\processed\orders_standardized.csv


**`DECISION_LOG.md`:** exact country-name corrections applied (list them), and how the Category Id/Name mismatch was resolved.
